This Spark Declarative Pipeline forms a key component of the Master Data Management (MDM) workflow. It transforms the raw Bronze layer data by standardizing physician addresses and phone numbers, applying consistent data-quality rules, and publishing the resulting curated datasets to the Silver schema. This creates a cleaner, more standardized foundation for downstream matching, deduplication, and analytics.


In [0]:
from pyspark import pipelines as dp
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, StructType, StructField
import re

# Address standardization UDF
@F.udf(returnType=StructType([
    StructField("line_1", StringType(), True),
    StructField("line_2", StringType(), True)
]))
def standardize_address(addr):
    """Split and standardize address into line 1 (street) and line 2 (suite/apt)"""
    if not addr:
        return ("", "")
    
    addr = addr.strip().upper()
    
    # Split pattern for suite/apt/floor details
    split_pattern = r'\b(SUITE|STE|APT|UNIT|BLDG|BUILDING|FLOOR|FL|#)\b'
    match = re.search(split_pattern, addr)
    
    if match:
        cutoff_idx = match.start()
        line_1 = addr[:cutoff_idx].strip().rstrip(",")
        line_2 = addr[cutoff_idx:].strip()
    else:
        line_1 = addr
        line_2 = ""
    
    # Standardize street types in line 1
    street_type_mapping = {
        'ST': 'Street', 'STR': 'Street', 'STREET': 'Street', 'ST.,': 'Street',
        'AVE': 'Avenue', 'AV': 'Avenue', 'AVENUE': 'Avenue',
        'RD': 'Road', 'RD.': 'Road', 'ROAD': 'Road',
        'BLVD': 'Boulevard', 'BLVD.': 'Boulevard', 'BOULEVARD': 'Boulevard',
        'DR': 'Drive', 'DR.': 'Drive', 'DRIVE': 'Drive',
        'LN': 'Lane', 'LN.': 'Lane', 'LANE': 'Lane',
        'CT': 'Court', 'CT.': 'Court', 'COURT': 'Court',
        'PL': 'Place', 'PL.': 'Place', 'PLACE': 'Place', 'PLZ': 'Plaza',
        'WAY': 'Way',
        'PKWY': 'Parkway', 'PKWY.': 'Parkway', 'PARKWAY': 'Parkway',
    }
    
    words = line_1.split()
    if words and words[-1] in street_type_mapping:
        words[-1] = street_type_mapping[words[-1]]
        line_1 = ' '.join(words)
    
    # Standardize directional indicators
    dir_mapping = {
        'N': 'NORTH', 'S': 'SOUTH', 'E': 'EAST', 'W': 'WEST',
        'NE': 'NORTHEAST', 'NW': 'NORTHWEST', 'SE': 'SOUTHEAST', 'SW': 'SOUTHWEST',
    }
    
    direct_indic = r'\b(N|NORTH|S|SOUTH|E|EAST|W|WEST|NE|NORTHEAST|NW|NORTHWEST|SE|SOUTHEAST|SW|SOUTHWEST)\b'
    match = re.search(direct_indic, line_1)
    if match:
        direction = match.group()
        if direction in dir_mapping:
            line_1 = re.sub(direct_indic, dir_mapping[direction], line_1)
    
    # Standardize line 2 (suite/apt abbreviations)
    addr_2_mapping = {
        'APT': 'APARTMENT', 'APT.': 'APARTMENT', 'APARTMENT': 'APARTMENT',
        'APT#': 'APARTMENT', 'APT.#': 'APARTMENT',
        'FL': 'FLOOR', 'FL.': 'FLOOR', 'FLOOR': 'FLOOR',
        'RM': 'ROOM', 'RM.': 'ROOM', 'ROOM': 'ROOM',
        'SUITE': 'SUITE', 'SUITE.': 'SUITE', 'STE': 'SUITE', 'STE.': 'SUITE',
        'UNIT': 'UNIT', 'UNIT.': 'UNIT',
        '#': 'SUITE'
    }
    
    pattern = "|".join(re.escape(k) for k in addr_2_mapping.keys())
    match = re.search(pattern, line_2)
    if match:
        addr_2 = match.group()
        line_2 = re.sub(pattern, addr_2_mapping.get(addr_2, addr_2), line_2)
    
    return (line_1.upper(), line_2)

# Phone cleaning UDF
@F.udf(returnType=StructType([
    StructField("phone_std", StringType(), True),
    StructField("phone_ext", StringType(), True)
]))
def clean_phone(phone):
    """Extract extension and standardize phone to 10-digit format"""
    if not phone:
        return ("", "")
    
    raw = str(phone)
    
    # Extract extension
    ext_match = re.search(r'(ext\.?|x|#)\s*(\d+)$', raw, flags=re.I)
    ext = ext_match.group(2) if ext_match else ""
    
    # Remove extension from main number
    raw = re.sub(r'(ext\.?|x|#)\s*\d+$', '', raw, flags=re.I)
    
    # Remove non-digits
    digits = re.sub(r'\D', '', raw)
    
    # Normalize US numbers (remove leading 1 from 11-digit numbers)
    if len(digits) == 11 and digits.startswith('1'):
        digits = digits[1:]
    
    # Keep only if 10 digits
    phone_std = digits if len(digits) == 10 else ""
    
    return (phone_std, ext)

@dp.table(
    name="cms_mdm_silver.cms_mdm_silver.silver_streaming_table",
    comment="Silver table with standardized doctor information from bronze layer"
)
def doctor_list_silver():
    """Transform bronze doctor data with address and phone standardization"""
    
    # Read bronze table
    df = spark.readStream.table("cms_mdm_bronze.cms_mdm_ingest.ca_doctor_list")
    
    # Apply address standardization
    df = df.withColumn("address_parts", standardize_address(F.col("Practice_Address")))
    df = df.withColumn("Practice_Address_line_1", F.col("address_parts.line_1"))
    df = df.withColumn("Practice_Address_line_2", F.col("address_parts.line_2"))
    df = df.drop("address_parts")
    
    # Apply phone cleaning
    df = df.withColumn("phone_parts", clean_phone(F.col("Phone")))
    df = df.withColumn("Phone_std", F.col("phone_parts.phone_std"))
    df = df.withColumn("Phone_Ext", F.col("phone_parts.phone_ext"))
    df = df.drop("phone_parts")
    
    # Select final columns in desired order
    df = df.select(
        "Legacy_System_ID",
        "NPI",
        "First_Name",
        "Last_Name",
        "Practice_Address",
        "Practice_Address_line_1",
        "Practice_Address_line_2",
        "City",
        "State",
        "Zip",
        "Phone",
        "Phone_std",
        "Phone_Ext",
        "Last_Updated_Date",
        "Source_System", 
        "Ingestion_Time", 
        "Ingestion_Date"
    )
    
    return df